# 01 — 2.5D EfficientNet-B4 Multi-Label Training
**RSNA Intracranial Hemorrhage Detection — Improvement Project**

### Reviewer-requested upgrades implemented here
| # | Upgrade | Implementation |
|---|---------|----------------|
| 1 | **2.5D input** | Stack prev/center/next slices → 9-channel input |
| 2 | **Multi-label** | 6 outputs (any + 5 subtypes) with weighted BCE |
| 3 | **EfficientNet-B4** | 19M params, first conv modified for 9ch |
| 4 | **Hard neg. mining** | Top-500 FP negatives 3× oversampled every 5 epochs |
| 5 | **5-fold GroupKFold CV** | Zero patient leakage, 5 fold models saved |
| 6 | **MixUp / CutMix** | Applied after warmup for regularization |
| 7 | **Label smoothing** | 0.05 to soften overconfident targets |

### Inputs required
1. **NB00 output** → `manifest_2_5d.csv` (image IDs, series adjacency, fold labels)
2. **NB02 output** → `cache/` (NPY files) + `normalization_stats.json`

### Session chaining
Training 5 folds takes ~15-25 hours.<br>
Each Kaggle session ≈ 12h → **plan for 2-3 sessions**.<br>
Set `PREV_SESSION_DIR` to resume from a committed session output.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  CONFIG  ██
# ══════════════════════════════════════════════════════════════════════════
import os, sys, json, random, shutil, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ─── Paths ────────────────────────────────────────────────────────────
NB00_DIR = Path('/kaggle/input/notebooks/harshitghosh/00-preprocess-metadata')
NB02_DIR = Path('/kaggle/input/notebooks/harshitghosh/nb02eda')

MANIFEST_PATH   = NB00_DIR / 'manifest_2_5d.csv'
NPY_CACHE_DIR   = NB02_DIR / 'cache'
NORM_STATS_PATH = NB02_DIR / 'normalization_stats.json'
SAVE_DIR        = Path('/kaggle/working')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ─── Session chaining ─────────────────────────────────────────────────
# Set this when resuming from a PREVIOUS session's committed output
PREV_SESSION_DIR = None
# e.g. Path('/kaggle/input/nb01-training-session-1/')
# The script will skip already-completed folds and copy their models.

# ─── Hyperparameters ──────────────────────────────────────────────────
BACKBONE     = 'tf_efficientnet_b4'
IMG_SIZE     = 256
IN_CHANNELS  = 9          # 3 windows × 3 slices (2.5D)
N_CLASSES    = 6          # any + 5 subtypes
BS           = 16         # GPU batch size
ACCUM_STEPS  = 4          # Effective batch = BS × ACCUM_STEPS = 64
WARMUP_EP    = 3          # Backbone frozen
TOTAL_EP     = 20         # Total epochs per fold
LR_WARMUP    = 1e-3       # LR for classifier during warmup
LR_MAIN      = 3e-4       # LR for full model after unfreeze
WEIGHT_DECAY = 1e-4
PATIENCE     = 5          # Early stopping patience (val AUC)
GRAD_CLIP    = 1.0
MINE_EVERY   = 5          # Hard-neg refresh interval (epochs)
MINE_TOP_K   = 500        # Number of hard negatives
MINE_BOOST   = 3.0        # Sampling weight multiplier
MIXUP_ALPHA  = 0.4
CUTMIX_ALPHA = 1.0
LABEL_SMOOTH = 0.05
ANY_WEIGHT   = 1.0        # Loss weight for 'any' column
SUB_WEIGHT   = 0.4        # Loss weight for subtype columns
NUM_WORKERS  = 4
N_FOLDS      = 5
SEED         = 42

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name()}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LOAD MANIFEST + NORMALIZATION  ██
# ══════════════════════════════════════════════════════════════════════════
manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest: {len(manifest):,} images, {manifest["patient_id"].nunique():,} patients')
print(f'Folds  : {sorted(manifest["fold"].unique())}')
print(f'Columns: {list(manifest.columns)}')

# Normalization stats (from NB02)
if NORM_STATS_PATH.exists():
    with open(NORM_STATS_PATH) as f:
        nstats = json.load(f)
    mean_3ch = np.array(nstats['mean'], dtype=np.float32)
    std_3ch  = np.array(nstats['std'],  dtype=np.float32)
else:
    print('WARNING: normalization_stats.json not found, using ImageNet defaults')
    mean_3ch = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std_3ch  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Tile to 9 channels: [brain, subdural, soft] × [prev, center, next]
MEAN_9CH = np.tile(mean_3ch, 3)
STD_9CH  = np.tile(std_3ch, 3)
print(f'\nNormalization (per window ch): mean={mean_3ch}, std={std_3ch}')

# Session chaining: determine which folds to train
START_FOLD = 0
if PREV_SESSION_DIR is not None:
    prev_state_path = Path(PREV_SESSION_DIR) / 'session_state.json'
    if prev_state_path.exists():
        prev_state = json.load(open(prev_state_path))
        START_FOLD = prev_state['next_fold']
        print(f'\nResuming from session: folds 0..{START_FOLD-1} already done')
        # Copy completed fold models
        for k in range(START_FOLD):
            src = Path(PREV_SESSION_DIR) / f'best_model_fold{k}.pth'
            dst = SAVE_DIR / f'best_model_fold{k}.pth'
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
                print(f'  Copied best_model_fold{k}.pth')
        # Copy metrics
        for k in range(START_FOLD):
            src = Path(PREV_SESSION_DIR) / f'metrics_fold{k}.csv'
            dst = SAVE_DIR / f'metrics_fold{k}.csv'
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
    else:
        print(f'WARNING: session_state.json not found in {PREV_SESSION_DIR}')

print(f'\nWill train folds: {list(range(START_FOLD, N_FOLDS))}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  DATASET: 2.5D  ██
# ══════════════════════════════════════════════════════════════════════════

class ICH2_5DDataset(Dataset):
    """Loads 3 adjacent slices (prev, center, next) → [H, W, 9] for 2.5D."""

    def __init__(self, df, cache_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.cache_dir = Path(cache_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _load_npy(self, image_id):
        """Load a single NPY slice; zeros if missing or NaN."""
        if pd.isna(image_id):
            return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        p = self.cache_dir / f'{image_id}.npy'
        if p.exists():
            arr = np.load(str(p))
            if arr.shape[:2] != (IMG_SIZE, IMG_SIZE):
                import cv2
                arr = cv2.resize(arr, (IMG_SIZE, IMG_SIZE))
            return arr.astype(np.uint8)
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Load prev / center / next slices
        prev_img   = self._load_npy(row.get('prev_image_id'))
        center_img = self._load_npy(row['image_id'])
        next_img   = self._load_npy(row.get('next_image_id'))

        # Concat → [H, W, 9]
        img_9ch = np.concatenate([prev_img, center_img, next_img], axis=2)

        if self.transform:
            img_9ch = self.transform(image=img_9ch)['image']  # → [9, H, W] tensor

        labels = torch.tensor([row[c] for c in SUBTYPES], dtype=torch.float32)
        return {'image': img_9ch, 'labels': labels, 'index': idx}


# ─── Transforms (albumentations handles any channel count) ─────────
def get_transforms(phase='train'):
    if phase == 'train':
        return A.Compose([
            A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.8, 1.0),
                                ratio=(0.9, 1.1), p=1.0),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                               rotate_limit=15, p=0.5,
                               border_mode=0),
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                            fill_value=0, p=0.3),
            A.Normalize(mean=MEAN_9CH.tolist(), std=STD_9CH.tolist(),
                        max_pixel_value=255.0),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Normalize(mean=MEAN_9CH.tolist(), std=STD_9CH.tolist(),
                        max_pixel_value=255.0),
            ToTensorV2(),
        ])

print('Dataset & transforms ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  MODEL: 2.5D EfficientNet-B4  ██
# ══════════════════════════════════════════════════════════════════════════

class ICHModel2_5D(nn.Module):
    def __init__(self, backbone_name=BACKBONE, n_classes=N_CLASSES,
                 in_channels=IN_CHANNELS, pretrained=True):
        super().__init__()
        # Load pretrained backbone (classifier removed)
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, drop_rate=0.3,
                                          drop_path_rate=0.2)

        # Modify first conv: 3ch → 9ch
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(
            in_channels, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None)
        )
        # Initialize: repeat pretrained 3ch weights k times, scale by 1/k
        k = in_channels // 3
        with torch.no_grad():
            new_conv.weight.copy_(
                old_conv.weight.repeat(1, k, 1, 1) / k
            )
            if old_conv.bias is not None:
                new_conv.bias.copy_(old_conv.bias)
        self.backbone.conv_stem = new_conv

        # Classifier head
        n_feat = self.backbone.num_features  # 1792 for B4
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_feat, n_classes)
        )

    def forward(self, x):
        features = self.backbone(x)  # [B, 1792]
        return self.classifier(features)  # [B, 6]

    def freeze_backbone(self):
        """Freeze backbone except conv_stem (new 9ch weights)."""
        for name, p in self.backbone.named_parameters():
            p.requires_grad = 'conv_stem' in name
        for p in self.classifier.parameters():
            p.requires_grad = True

    def unfreeze_backbone(self):
        for p in self.parameters():
            p.requires_grad = True


# Quick test
_m = ICHModel2_5D(pretrained=False)
n_params = sum(p.numel() for p in _m.parameters()) / 1e6
print(f'{BACKBONE} 2.5D: {n_params:.1f}M params')
print(f'  conv_stem: {_m.backbone.conv_stem}')
print(f'  head features: {_m.backbone.num_features}')
with torch.no_grad():
    _out = _m(torch.randn(2, IN_CHANNELS, IMG_SIZE, IMG_SIZE))
    print(f'  test forward: input [2,{IN_CHANNELS},{IMG_SIZE},{IMG_SIZE}] → output {list(_out.shape)}')
del _m

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LOSS + MIXUP / CUTMIX  ██
# ══════════════════════════════════════════════════════════════════════════

class MultiLabelLoss(nn.Module):
    """Weighted BCE with label smoothing. 'any' gets higher weight."""
    def __init__(self, any_w=ANY_WEIGHT, sub_w=SUB_WEIGHT, ls=LABEL_SMOOTH):
        super().__init__()
        self.any_w = any_w
        self.sub_w = sub_w
        self.ls = ls

    def forward(self, logits, targets):
        # Label smoothing
        t = targets * (1 - self.ls) + 0.5 * self.ls
        bce = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
        # Per-column weights
        w = torch.ones_like(bce)
        w[:, 0] = self.any_w
        w[:, 1:] = self.sub_w
        return (bce * w).mean()


# ─── MixUp / CutMix ──────────────────────────────────────────────────
def rand_bbox(size, lam):
    """Random bounding box for CutMix."""
    H, W = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_h, cut_w = int(H * cut_rat), int(W * cut_rat)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    return y1, x1, y2, x2


def apply_mix(images, targets, epoch, warmup_ep=WARMUP_EP,
              mixup_a=MIXUP_ALPHA, cutmix_a=CUTMIX_ALPHA, p=0.5):
    """Apply MixUp or CutMix (only after warmup).
    Returns: mixed_images, targets_a, targets_b, lam
    """
    if epoch < warmup_ep or random.random() > p:
        return images, targets, targets, 1.0

    idx = torch.randperm(images.size(0), device=images.device)

    if random.random() < 0.5:  # MixUp
        lam = np.random.beta(mixup_a, mixup_a)
        images = lam * images + (1 - lam) * images[idx]
    else:  # CutMix
        lam = np.random.beta(cutmix_a, cutmix_a)
        y1, x1, y2, x2 = rand_bbox(images.size(), lam)
        images[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
        lam = 1 - (y2 - y1) * (x2 - x1) / (images.size(-2) * images.size(-1))

    return images, targets, targets[idx], lam


print('Loss + augmentation utilities ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HARD NEGATIVE MINING  ██
# ══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def mine_hard_negatives(model, dataset, device, top_k=MINE_TOP_K):
    """Forward pass on training set → find top-K false-positive negatives.
    Returns a set of dataset indices to oversample.
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=BS * 2, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    all_probs, all_labels, all_idxs = [], [], []

    for batch in tqdm(loader, desc='Hard-neg mining', leave=False):
        imgs = batch['image'].to(device, non_blocking=True)
        with autocast():
            logits = model(imgs)
        probs = torch.sigmoid(logits[:, 0]).cpu()  # P(any)
        all_probs.append(probs)
        all_labels.append(batch['labels'][:, 0])
        all_idxs.append(batch['index'])

    probs  = torch.cat(all_probs)
    labels = torch.cat(all_labels)
    idxs   = torch.cat(all_idxs)

    # False positives: label=0 but high P(any)
    neg_mask   = labels < 0.5
    neg_probs  = probs[neg_mask]
    neg_idxs   = idxs[neg_mask]
    k = min(top_k, len(neg_probs))
    _, top_pos = neg_probs.topk(k)
    hard_set = set(neg_idxs[top_pos].tolist())

    print(f'  Mined {len(hard_set)} hard negatives '
          f'(max FP prob={neg_probs[top_pos[0]]:.3f})')
    return hard_set


def build_sampler(n_samples, hard_neg_indices=None, boost=MINE_BOOST):
    """WeightedRandomSampler: hard negatives get boosted weight."""
    weights = np.ones(n_samples, dtype=np.float64)
    if hard_neg_indices:
        for idx in hard_neg_indices:
            if 0 <= idx < n_samples:
                weights[idx] *= boost
    return WeightedRandomSampler(
        weights=weights, num_samples=n_samples, replacement=True
    )

print('Hard-negative mining ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  TRAIN / VALIDATE EPOCH  ██
# ══════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, scheduler,
                    scaler, device, epoch):
    model.train()
    running_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'  Train ep{epoch:02d}', leave=False)
    for step, batch in enumerate(pbar):
        imgs   = batch['image'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        # MixUp / CutMix
        imgs, targets_a, targets_b, lam = apply_mix(imgs, labels, epoch)

        with autocast():
            logits = model(imgs)
            loss = (lam * criterion(logits, targets_a)
                    + (1 - lam) * criterion(logits, targets_b))
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        running_loss += loss.item() * ACCUM_STEPS
        n_batches += 1
        pbar.set_postfix(loss=f'{running_loss/n_batches:.4f}')

    return running_loss / max(n_batches, 1)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    n_batches = 0
    all_probs, all_labels = [], []

    for batch in tqdm(loader, desc='  Val', leave=False):
        imgs   = batch['image'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)

        running_loss += loss.item()
        n_batches += 1

        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)
        all_labels.append(batch['labels'])

    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()

    # Per-class AUC
    aucs = {}
    for i, name in enumerate(SUBTYPES):
        try:
            aucs[name] = roc_auc_score(labels[:, i], probs[:, i])
        except ValueError:
            aucs[name] = 0.5  # Single class in fold
    aucs['mean'] = np.mean(list(aucs.values()))

    return running_loss / max(n_batches, 1), aucs

print('Train/val functions ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  TRAIN ONE FOLD  ██
# ══════════════════════════════════════════════════════════════════════════

def train_fold(fold_id, manifest, save_dir):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold_id}/{N_FOLDS-1}')
    print(f'{"="*60}')
    fold_start = time.time()

    # ── Split ──────────────────────────────────────────────────────
    train_df = manifest[manifest['fold'] != fold_id]
    val_df   = manifest[manifest['fold'] == fold_id]
    print(f'  Train: {len(train_df):,}  Val: {len(val_df):,}')

    # ── Datasets ───────────────────────────────────────────────────
    train_ds = ICH2_5DDataset(train_df, NPY_CACHE_DIR, get_transforms('train'))
    val_ds   = ICH2_5DDataset(val_df,   NPY_CACHE_DIR, get_transforms('val'))

    val_loader = DataLoader(val_ds, batch_size=BS * 2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True,
                            drop_last=False)

    # ── Model, loss, scaler ────────────────────────────────────────
    model = ICHModel2_5D(pretrained=True).to(DEVICE)
    criterion = MultiLabelLoss()
    scaler = GradScaler()

    best_auc = 0.0
    patience_counter = 0
    metrics_log = []
    hard_neg_set = None

    for epoch in range(TOTAL_EP):
        ep_start = time.time()

        # ── Phase control ──────────────────────────────────────────
        if epoch < WARMUP_EP:
            model.freeze_backbone()
            trainable = [p for p in model.parameters() if p.requires_grad]
            optimizer = torch.optim.Adam(trainable, lr=LR_WARMUP)
            scheduler = None
            phase_name = 'warmup'
        elif epoch == WARMUP_EP:
            model.unfreeze_backbone()
            optimizer = torch.optim.AdamW(model.parameters(),
                                          lr=LR_MAIN, weight_decay=WEIGHT_DECAY)
            steps_per_ep = len(train_ds) // (BS * ACCUM_STEPS)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=steps_per_ep * (TOTAL_EP - WARMUP_EP))
            phase_name = 'main (just unfrozen)'
        else:
            phase_name = 'main'

        # ── Hard negative mining refresh ───────────────────────────
        if (epoch >= WARMUP_EP and epoch % MINE_EVERY == 0):
            hard_neg_set = mine_hard_negatives(model, train_ds, DEVICE)

        # ── Build train loader (with mining weights if available) ──
        sampler = build_sampler(len(train_ds), hard_neg_set)
        train_loader = DataLoader(
            train_ds, batch_size=BS, sampler=sampler,
            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

        # ── Train ──────────────────────────────────────────────────
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler,
            scaler, DEVICE, epoch)

        # ── Validate ───────────────────────────────────────────────
        val_loss, val_aucs = validate(model, val_loader, criterion, DEVICE)

        ep_time = time.time() - ep_start
        lr_now = optimizer.param_groups[0]['lr']

        row = {'fold': fold_id, 'epoch': epoch, 'phase': phase_name,
               'train_loss': train_loss, 'val_loss': val_loss,
               'lr': lr_now, 'time_min': ep_time / 60}
        for k, v in val_aucs.items():
            row[f'auc_{k}'] = v
        metrics_log.append(row)

        print(f'  Ep{epoch:02d} [{phase_name:18s}] '
              f'trn={train_loss:.4f}  val={val_loss:.4f}  '
              f'AUC_any={val_aucs["any"]:.4f}  AUC_mean={val_aucs["mean"]:.4f}  '
              f'LR={lr_now:.1e}  {ep_time/60:.1f}min')

        # ── Best model + early stopping ────────────────────────────
        if val_aucs['mean'] > best_auc:
            best_auc = val_aucs['mean']
            patience_counter = 0
            torch.save(model.state_dict(),
                       save_dir / f'best_model_fold{fold_id}.pth')
            print(f'    >>> New best AUC_mean={best_auc:.5f} — saved')
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE and epoch >= WARMUP_EP:
                print(f'    Early stopping at epoch {epoch}')
                break

    # ── Save fold metrics ──────────────────────────────────────────
    pd.DataFrame(metrics_log).to_csv(
        save_dir / f'metrics_fold{fold_id}.csv', index=False)

    fold_time = (time.time() - fold_start) / 3600
    print(f'\n  Fold {fold_id} done — best AUC_mean={best_auc:.5f} — '
          f'{fold_time:.1f}h')

    # Clean up GPU
    del model, optimizer, scaler, train_ds, val_ds
    torch.cuda.empty_cache()

    return best_auc

print('train_fold() ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  5-FOLD TRAINING LOOP  ██
# ══════════════════════════════════════════════════════════════════════════

fold_results = {}
session_start = time.time()

for fold_id in range(START_FOLD, N_FOLDS):
    best_auc = train_fold(fold_id, manifest, SAVE_DIR)
    fold_results[fold_id] = best_auc

    # ── Save session state after each fold (for chaining) ──────────
    state = {
        'completed_folds': list(fold_results.keys()),
        'next_fold': fold_id + 1,
        'fold_aucs': {str(k): v for k, v in fold_results.items()},
        'backbone': BACKBONE,
        'in_channels': IN_CHANNELS,
        'n_classes': N_CLASSES,
    }
    with open(SAVE_DIR / 'session_state.json', 'w') as f:
        json.dump(state, f, indent=2)

    elapsed_h = (time.time() - session_start) / 3600
    print(f'\nSession time so far: {elapsed_h:.1f}h')

    # Auto-stop if nearing Kaggle 12h limit
    if elapsed_h > 10.5 and fold_id < N_FOLDS - 1:
        print(f'\n  STOPPING: approaching 12h Kaggle limit.')
        print(f'  Completed folds: {list(fold_results.keys())}')
        print(f'  Commit this notebook, then create a new session with:')
        print(f'    PREV_SESSION_DIR = Path(\'<this notebook output path>\')')
        break

print(f'\n{"="*60}')
print('SESSION COMPLETE')
print(f'{"="*60}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  SUMMARY  ██
# ══════════════════════════════════════════════════════════════════════════

# Fold AUC summary
print('Per-fold best AUC (mean over 6 classes):')
auc_vals = []
for k in sorted(fold_results.keys()):
    v = fold_results[k]
    print(f'  Fold {k}: {v:.5f}')
    auc_vals.append(v)
if len(auc_vals) > 1:
    print(f'  ───────────────')
    print(f'  Mean  : {np.mean(auc_vals):.5f}')
    print(f'  Std   : {np.std(auc_vals):.5f}')

# Disk usage
saved_files = sorted(SAVE_DIR.glob('*'))
total_mb = 0
print(f'\nSaved files in {SAVE_DIR}:')
for f in saved_files:
    sz = f.stat().st_size / 1e6
    total_mb += sz
    print(f'  {f.name:40s} {sz:>8.1f} MB')
print(f'  {"TOTAL":40s} {total_mb:>8.1f} MB')

# Next steps
if max(fold_results.keys()) < N_FOLDS - 1:
    print(f'\n▸ Not all folds done. Commit & resume with PREV_SESSION_DIR.')
else:
    print(f'\n▸ All {N_FOLDS} folds complete!')
    print(f'  Next: run 02_improved_eval_report.ipynb')
    print(f'  Add this notebook output + NB00 output + NB02 output as inputs.')

# Improved Training — 5-Fold CV, EfficientNet-B4, Multi-Label
**RSNA Intracranial Hemorrhage Detection — Improvement Project**

### Reviewer credibility progression
| Scenario | Maximum Claim |
|----------|---------------|
| Current (single split, B0 binary) | ~95% AUC, 90% sensitivity, pilot results |
| +B4 backbone | 96% AUC slice-level, 91-92% sensitivity |
| +Multi-label | 96-97% AUC patient-level, 93-94% sensitivity |
| **+5-fold CV** | **96.5 ± 0.5% AUC, 93 ± 2% sensitivity, hospital-ready** |

This notebook implements **all four tiers** simultaneously.

### What changed vs baseline (NB03)
| # | Component | Baseline | Improved |
|---|-----------|----------|----------|
| 1 | Backbone | EfficientNet-B0 (5.3M) | **EfficientNet-B4 (19.3M)** |
| 2 | Labels | Binary (`any` only) | **Multi-label (6 outputs)** |
| 3 | Validation | Single 85/15 split | **GroupKFold 5-fold CV** |
| 4 | Loss | BCE + pos_weight | **Weighted multi-label + label smoothing** |
| 5 | Augmentation | Flip + Rotation + ColorJitter | **+ MixUp + CutMix + Affine + Erasing** |
| 6 | Scheduler | CosineAnnealing | **OneCycleLR** |
| 7 | Batch size | 16 | **16 x 4 grad-accum = effective 64** |

### Session-chaining workflow (Kaggle free tier, 20 GB)
| Session | What happens |
|---------|-------------|
| 1 | Train folds 0-1 (maybe 2 if fast) |
| 2+ | Resume from checkpoint, finish remaining folds |

> **Before each re-run**: set `PREV_CHECKPOINT_DIR` to the output of the
> previous session. The code auto-detects completed folds and resumes.

### Storage budget (20 GB limit)
| Item | Size | Total |
|------|------|-------|
| best_model_fold{k}.pth (x5) | ~77 MB | ~385 MB |
| checkpoint_fold{k}.pth (x1) | ~230 MB | ~230 MB |
| fold_splits.json + metrics CSVs | <1 MB | <1 MB |
| **Total peak** | | **~616 MB << 20 GB** |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  CONFIGURATION  ██
# ══════════════════════════════════════════════════════════════════════════
import os, gc, time, json, random, shutil
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ─── Paths ────────────────────────────────────────────────────────────
CACHE_INPUT_DIR = Path('/kaggle/input/notebooks/harshitghosh/nb02eda')
NPY_CACHE_DIR   = CACHE_INPUT_DIR / 'cache'
MANIFEST_PATH   = CACHE_INPUT_DIR / 'manifest.csv'
STATS_PATH      = CACHE_INPUT_DIR / 'normalization_stats.json'

# Session chaining: set this to the output of the PREVIOUS session
# Set to None for the very first session
PREV_CHECKPOINT_DIR = None
# PREV_CHECKPOINT_DIR = '/kaggle/input/notebooks/harshitghosh/01-improved-training'

SAVE_DIR = Path('/kaggle/working')

# ─── Architecture ─────────────────────────────────────────────────────
ARCH          = 'efficientnet_b4'
N_CLASSES     = 6
LABEL_COLS    = ['any', 'epidural', 'intraparenchymal',
                 'intraventricular', 'subarachnoid', 'subdural']

# ─── Training ─────────────────────────────────────────────────────────
N_FOLDS       = 5
TOTAL_EPOCHS  = 20
BATCH_SIZE    = 16
ACCUM_STEPS   = 4          # effective batch = 64
BASE_LR       = 3e-4
BACKBONE_LR_MULT = 0.1    # backbone gets LR * 0.1
WEIGHT_DECAY  = 1e-4
DROPOUT       = 0.5
FREEZE_EPOCHS = 3          # freeze backbone first N epochs
PATIENCE      = 7          # early stopping patience
LABEL_SMOOTH  = 0.05
AUX_WEIGHT    = 0.4        # weight for subtype losses
MIXUP_ALPHA   = 0.4
CUTMIX_ALPHA  = 1.0
IMG_SIZE      = 256
NUM_WORKERS   = 4
SEED          = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == 'cuda': torch.cuda.manual_seed_all(SEED)

print(f'Device       : {DEVICE}')
print(f'Architecture : {ARCH}')
print(f'Folds        : {N_FOLDS}')
print(f'Eff. batch   : {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE * ACCUM_STEPS}')
print(f'Max epochs   : {TOTAL_EPOCHS} per fold')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  VERIFY NB02 CACHE + LOAD MANIFEST  ██
# ══════════════════════════════════════════════════════════════════════════
assert CACHE_INPUT_DIR.exists(), f'Cache dir not found: {CACHE_INPUT_DIR}'
assert MANIFEST_PATH.exists(),   f'manifest.csv not found: {MANIFEST_PATH}'
assert STATS_PATH.exists(),      f'normalization_stats.json not found'

npy_count = len(list(NPY_CACHE_DIR.glob('*.npy')))
assert npy_count > 0, 'No NPY files in cache'
print(f'NPY files in cache: {npy_count:,}')

with open(STATS_PATH) as f:
    norm_stats = json.load(f)
MEAN = norm_stats['mean']
STD  = norm_stats['std']
print(f'Norm stats: mean={MEAN}, std={STD}')

# Load manifest — use ALL data (not just original train split)
manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest rows: {len(manifest):,}')

# Ensure all 6 label columns exist
for col in LABEL_COLS:
    if col not in manifest.columns:
        print(f'  Warning: column "{col}" missing, filling with 0')
        manifest[col] = 0

print(f'Label distribution:')
for col in LABEL_COLS:
    n_pos = manifest[col].sum()
    print(f'  {col:25s}: {n_pos:>7,} / {len(manifest):,} ({100*n_pos/len(manifest):.1f}%)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  5-FOLD SPLITS (GroupKFold by patient_id)  ██
# ══════════════════════════════════════════════════════════════════════════

def load_or_create_splits(manifest, n_folds, save_dir, prev_dir=None):
    """Load existing fold_splits.json or create from scratch."""
    fname = 'fold_splits.json'
    local_path = save_dir / fname

    # 1. Check current working directory
    if local_path.exists():
        with open(local_path) as f:
            data = json.load(f)
        print(f'Loaded fold splits from {local_path}')
        return data['folds']

    # 2. Check chained input from previous session
    if prev_dir is not None:
        chained = Path(prev_dir) / fname
        if chained.exists():
            shutil.copy2(chained, local_path)
            with open(local_path) as f:
                data = json.load(f)
            print(f'Copied fold splits from chained session: {chained}')
            return data['folds']

    # 3. Create fresh splits
    gkf = GroupKFold(n_splits=n_folds)
    groups = manifest['patient_id'].values
    folds = []
    for train_idx, val_idx in gkf.split(manifest, groups=groups):
        folds.append({'train': train_idx.tolist(), 'val': val_idx.tolist()})

    data = {'n_folds': n_folds, 'seed': SEED, 'folds': folds}
    with open(local_path, 'w') as f:
        json.dump(data, f)
    print(f'Created {n_folds}-fold GroupKFold splits → {local_path}')
    return folds


fold_indices = load_or_create_splits(
    manifest, N_FOLDS, SAVE_DIR, PREV_CHECKPOINT_DIR
)

# Print fold sizes
print(f'\n{"Fold":>5s}  {"Train":>10s}  {"Val":>10s}  {"Val patients":>13s}')
for k in range(N_FOLDS):
    t_n = len(fold_indices[k]['train'])
    v_n = len(fold_indices[k]['val'])
    v_pat = manifest.iloc[fold_indices[k]['val']]['patient_id'].nunique()
    print(f'{k:>5d}  {t_n:>10,}  {v_n:>10,}  {v_pat:>13,}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  MODEL DEFINITION  ██
# ══════════════════════════════════════════════════════════════════════════

class MultiLabelICHModel(nn.Module):
    """EfficientNet backbone → dropout → 6-output multi-label head."""
    _ARCHS = {
        'efficientnet_b2': models.efficientnet_b2,
        'efficientnet_b3': models.efficientnet_b3,
        'efficientnet_b4': models.efficientnet_b4,
    }

    def __init__(self, arch, n_classes=6, dropout=0.5):
        super().__init__()
        factory = self._ARCHS[arch]
        backbone = factory(weights='IMAGENET1K_V1')
        in_feat = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(in_feat, n_classes)

    def forward(self, x):
        return self.fc(self.drop(self.backbone(x)))


# Quick param count
_tmp = MultiLabelICHModel(ARCH, N_CLASSES, DROPOUT)
n_params = sum(p.numel() for p in _tmp.parameters())
print(f'{ARCH}: {n_params/1e6:.1f}M parameters')
del _tmp

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  DATASET + TRANSFORMS  ██
# ══════════════════════════════════════════════════════════════════════════

class ICHDatasetML(Dataset):
    """Multi-label dataset: returns 6-dim target vector."""
    def __init__(self, df, npy_root, label_cols, transform):
        self.df = df.reset_index(drop=True)
        self.npy_root = npy_root
        self.label_cols = label_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.npy_root, f'{row["image_id"]}.npy')
        try:
            img = np.load(path)
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        label = np.array([row[c] for c in self.label_cols], dtype=np.float32)
        return self.transform(img), label


train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
    T.RandomErasing(p=0.15, scale=(0.02, 0.15)),
])

val_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

print('Transforms ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LOSS + MIXUP + CUTMIX  ██
# ══════════════════════════════════════════════════════════════════════════

class MultiLabelLoss(nn.Module):
    """
    Primary 'any' loss (weight=1) + auxiliary subtype losses (weight=AUX_WEIGHT).
    Per-class pos_weight + label smoothing.
    """
    def __init__(self, pos_weights, label_smoothing=0.05, aux_weight=0.4):
        super().__init__()
        self.aux_weight = aux_weight
        self.ls = label_smoothing
        self.register_buffer('pw', torch.tensor(pos_weights, dtype=torch.float32))

    def forward(self, logits, targets):
        t = targets * (1.0 - self.ls) + 0.5 * self.ls  # label smoothing
        losses = []
        for i in range(logits.shape[1]):
            l = F.binary_cross_entropy_with_logits(
                logits[:, i], t[:, i], pos_weight=self.pw[i:i+1])
            losses.append(l)
        return losses[0] + self.aux_weight * sum(losses[1:]) / max(len(losses)-1, 1)


def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]


def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    H, W = x.shape[2], x.shape[3]
    r = np.sqrt(1 - lam)
    ch, cw = int(H * r), int(W * r)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1 = np.clip(cy - ch//2, 0, H); y2 = np.clip(cy + ch//2, 0, H)
    x1 = np.clip(cx - cw//2, 0, W); x2 = np.clip(cx + cw//2, 0, W)
    x_out = x.clone()
    x_out[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam_eff = 1 - (y2-y1)*(x2-x1)/(H*W)
    return x_out, lam_eff * y + (1 - lam_eff) * y[idx]


print('Loss + MixUp/CutMix ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  TRAINING & VALIDATION FUNCTIONS  ██
# ══════════════════════════════════════════════════════════════════════════

scaler = torch.amp.GradScaler()

def train_one_epoch(model, loader, optimizer, scheduler, criterion, epoch):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # MixUp or CutMix (50/50), only after backbone unfreeze
        if epoch >= FREEZE_EPOCHS:
            if np.random.random() < 0.5:
                imgs, labels = mixup_data(imgs, labels, MIXUP_ALPHA)
            else:
                imgs, labels = cutmix_data(imgs, labels, CUTMIX_ALPHA)

        with torch.amp.autocast(device_type='cuda'):
            loss = criterion(model(imgs), labels) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        scheduler.step()
        running_loss += loss.item() * ACCUM_STEPS

    return running_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    running_loss = 0.0

    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type='cuda'):
            logits = model(imgs)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = 1 / (1 + np.exp(-logits))

    # AUC on 'any' (col 0)
    auc = roc_auc_score(labels[:, 0], probs[:, 0])
    preds = (probs[:, 0] > 0.5).astype(int)
    f1 = f1_score(labels[:, 0].astype(int), preds, zero_division=0)

    return float(auc), float(f1)


print('Training functions ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  5-FOLD TRAINING LOOP (session-chainable)  ██
# ══════════════════════════════════════════════════════════════════════════

def find_artifact(filename):
    """Find file in current session or chained input."""
    local = SAVE_DIR / filename
    if local.exists():
        return str(local)
    if PREV_CHECKPOINT_DIR is not None:
        chained = Path(PREV_CHECKPOINT_DIR) / filename
        if chained.exists():
            shutil.copy2(chained, local)
            return str(local)
    return None


def fold_is_complete(fold_id):
    return find_artifact(f'best_model_fold{fold_id}.pth') is not None


# ── Status check ──────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('  SESSION STATUS')
print('=' * 65)
for k in range(N_FOLDS):
    status = 'COMPLETE' if fold_is_complete(k) else 'pending'
    ckpt = find_artifact(f'checkpoint_fold{k}.pth')
    if ckpt and not fold_is_complete(k):
        status = 'HAS CHECKPOINT (will resume)'
    print(f'  Fold {k}: {status}')

completed = sum(1 for k in range(N_FOLDS) if fold_is_complete(k))
remaining = N_FOLDS - completed
print(f'\n  Completed: {completed}/{N_FOLDS}   Remaining: {remaining}')
print('=' * 65)

# ── Main loop ─────────────────────────────────────────────────────────
fold_results = []
session_start = time.time()

for fold_id in range(N_FOLDS):

    best_path    = SAVE_DIR / f'best_model_fold{fold_id}.pth'
    ckpt_path    = SAVE_DIR / f'checkpoint_fold{fold_id}.pth'
    metrics_path = SAVE_DIR / f'metrics_fold{fold_id}.csv'

    # ── Skip completed folds ──────────────────────────────────────
    if fold_is_complete(fold_id):
        if metrics_path.exists():
            m = pd.read_csv(metrics_path)
            fold_results.append({'fold': fold_id,
                                 'best_auc': m['val_auc'].max(),
                                 'best_f1': m['val_f1'].max()})
        print(f'\n>>> Fold {fold_id}: already complete, skipping.\n')
        continue

    print(f'\n{"="*65}')
    print(f'  FOLD {fold_id} / {N_FOLDS-1}')
    print(f'{"="*65}')

    # ── Fold data ─────────────────────────────────────────────────
    train_df = manifest.iloc[fold_indices[fold_id]['train']].reset_index(drop=True)
    val_df   = manifest.iloc[fold_indices[fold_id]['val']].reset_index(drop=True)

    train_ds = ICHDatasetML(train_df, str(NPY_CACHE_DIR), LABEL_COLS, train_transform)
    val_ds   = ICHDatasetML(val_df,   str(NPY_CACHE_DIR), LABEL_COLS, val_transform)

    # Weighted sampler (by 'any' column)
    any_labels = train_df['any'].values
    n_pos = any_labels.sum(); n_neg = len(any_labels) - n_pos
    w_pos = len(any_labels) / (2 * max(n_pos, 1))
    w_neg = len(any_labels) / (2 * max(n_neg, 1))
    sample_weights = np.where(any_labels == 1, w_pos, w_neg)
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, BATCH_SIZE * 2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print(f'  Train: {len(train_df):,}  Val: {len(val_df):,}  (pos rate: {100*n_pos/len(train_df):.1f}%)')

    # ── Per-class pos_weight ──────────────────────────────────────
    pos_weights = []
    for col in LABEL_COLS:
        p = train_df[col].sum()
        pos_weights.append(max((len(train_df) - p) / max(p, 1), 1.0))
    print(f'  pos_weights: {[f"{w:.1f}" for w in pos_weights]}')

    # ── Model ─────────────────────────────────────────────────────
    model = MultiLabelICHModel(ARCH, N_CLASSES, DROPOUT).to(DEVICE)
    criterion = MultiLabelLoss(pos_weights, LABEL_SMOOTH, AUX_WEIGHT).to(DEVICE)

    backbone_params = list(model.backbone.parameters())
    head_params     = list(model.drop.parameters()) + list(model.fc.parameters())
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': BASE_LR * BACKBONE_LR_MULT},
        {'params': head_params,     'lr': BASE_LR},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = TOTAL_EPOCHS * len(train_loader)
    scheduler   = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[BASE_LR * BACKBONE_LR_MULT, BASE_LR],
        total_steps=total_steps)

    # ── Resume from checkpoint ────────────────────────────────────
    start_epoch = 0
    best_auc    = 0.0
    metrics_rows = []

    if ckpt_path.exists():
        ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch  = ckpt['epoch'] + 1
        best_auc     = ckpt['best_auc']
        metrics_rows = ckpt.get('metrics', [])
        print(f'  Resumed from epoch {start_epoch}, best_auc={best_auc:.5f}')

    # ── Backbone freeze ───────────────────────────────────────────
    if start_epoch < FREEZE_EPOCHS:
        for p in backbone_params:
            p.requires_grad = False
        print(f'  Backbone frozen for epochs 0-{FREEZE_EPOCHS-1}')

    # ── Epoch loop ────────────────────────────────────────────────
    patience_ctr = 0

    for epoch in range(start_epoch, TOTAL_EPOCHS):
        t0 = time.time()

        # Unfreeze backbone
        if epoch == FREEZE_EPOCHS:
            for p in backbone_params:
                p.requires_grad = True
            print(f'  >>> Backbone UNFROZEN at epoch {epoch}')

        train_loss = train_one_epoch(model, train_loader, optimizer,
                                     scheduler, criterion, epoch)
        val_auc, val_f1 = validate(model, val_loader)
        elapsed = time.time() - t0

        metrics_rows.append({
            'fold': fold_id, 'epoch': epoch,
            'train_loss': round(train_loss, 5),
            'val_auc': round(val_auc, 5),
            'val_f1': round(val_f1, 5),
            'time_s': round(elapsed, 1),
        })

        marker = ''
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), str(best_path))
            patience_ctr = 0
            marker = f' << NEW BEST'
        else:
            patience_ctr += 1

        print(f'  E{epoch:02d} | loss={train_loss:.4f} | AUC={val_auc:.5f} | '
              f'F1={val_f1:.4f} | {elapsed:.0f}s | pat={patience_ctr}{marker}')

        # Save checkpoint every epoch (for session resume)
        torch.save({
            'epoch': epoch, 'fold': fold_id,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_auc': best_auc,
            'metrics': metrics_rows,
        }, str(ckpt_path))

        # Early stopping
        if patience_ctr >= PATIENCE:
            print(f'  Early stopping (patience={PATIENCE})')
            break

    # ── Fold complete ─────────────────────────────────────────────
    pd.DataFrame(metrics_rows).to_csv(str(metrics_path), index=False)
    if ckpt_path.exists():
        os.remove(str(ckpt_path))  # save storage

    fold_results.append({'fold': fold_id, 'best_auc': best_auc,
                         'best_f1': max(r['val_f1'] for r in metrics_rows)})

    del model, optimizer, scheduler, criterion
    del train_loader, val_loader, train_ds, val_ds
    gc.collect(); torch.cuda.empty_cache()

    print(f'\n  Fold {fold_id} DONE  |  best AUC = {best_auc:.5f}')
    elapsed_total = (time.time() - session_start) / 3600
    print(f'  Session time so far: {elapsed_total:.1f} h')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  POST-TRAINING SUMMARY  ██
# ══════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 65)
print('  5-FOLD TRAINING SUMMARY')
print('=' * 65)

# Collect from saved metrics CSVs (works even on resume sessions)
fold_results_full = []
for k in range(N_FOLDS):
    mp = SAVE_DIR / f'metrics_fold{k}.csv'
    if mp.exists():
        m = pd.read_csv(mp)
        fold_results_full.append({
            'fold': k,
            'best_auc': m['val_auc'].max(),
            'best_f1': m['val_f1'].max(),
            'epochs': len(m),
        })
    elif fold_is_complete(k):
        fold_results_full.append({'fold': k, 'best_auc': 0, 'best_f1': 0, 'epochs': 0})
    else:
        print(f'  ⚠ Fold {k} NOT complete — rerun this notebook')

if fold_results_full:
    aucs = [r['best_auc'] for r in fold_results_full if r['best_auc'] > 0]
    f1s  = [r['best_f1']  for r in fold_results_full if r['best_f1'] > 0]

    print(f'\n  {"Fold":>5s}  {"Best AUC":>10s}  {"Best F1":>10s}  {"Epochs":>7s}')
    print(f'  {"─"*5}  {"─"*10}  {"─"*10}  {"─"*7}')
    for r in fold_results_full:
        print(f'  {r["fold"]:>5d}  {r["best_auc"]:>10.5f}  {r["best_f1"]:>10.4f}  {r["epochs"]:>7d}')

    if len(aucs) == N_FOLDS:
        mean_auc = np.mean(aucs)
        std_auc  = np.std(aucs)
        mean_f1  = np.mean(f1s)
        std_f1   = np.std(f1s)
        print(f'\n  Mean AUC : {mean_auc:.5f} +/- {std_auc:.5f}')
        print(f'  Mean F1  : {mean_f1:.4f} +/- {std_f1:.4f}')
        print(f'\n  Target    : 96.5 +/- 0.5% AUC (hospital-ready)')
        print(f'  Achieved  : {mean_auc*100:.1f} +/- {std_auc*100:.1f}%')
    else:
        print(f'\n  Only {len(aucs)}/{N_FOLDS} folds complete.')
        print(f'  Set PREV_CHECKPOINT_DIR to this output and rerun.')

# Storage report
total_mb = 0
for f in SAVE_DIR.glob('*'):
    total_mb += f.stat().st_size / 1e6
print(f'\n  Storage used: {total_mb:.0f} MB / 20,000 MB ({100*total_mb/20000:.1f}%)')
print('=' * 65)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  TRAINING CURVES (all folds)  ██
# ══════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.tab10.colors

for k in range(N_FOLDS):
    mp = SAVE_DIR / f'metrics_fold{k}.csv'
    if not mp.exists(): continue
    m = pd.read_csv(mp)
    c = colors[k]
    axes[0].plot(m['epoch'], m['train_loss'], '--', color=c, alpha=0.5)
    axes[0].plot(m['epoch'], m['val_auc'],    '-',  color=c, label=f'Fold {k} ({m["val_auc"].max():.4f})')
    # dummy for loss legend
    axes[1].plot(m['epoch'], m['val_f1'],     '-',  color=c, label=f'Fold {k}')

axes[0].set(title='AUC per Fold (solid) + Train Loss (dashed)',
            xlabel='Epoch', ylabel='AUC / Loss')
axes[0].legend(fontsize=8)
axes[1].set(title='F1 per Fold', xlabel='Epoch', ylabel='F1')
axes[1].legend(fontsize=8)

plt.suptitle(f'5-Fold Training Curves — {ARCH}', fontsize=13)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'fold_training_curves.png'), bbox_inches='tight')
plt.show()
print('Saved fold_training_curves.png')

## Next Steps

### If all 5 folds are complete:
Run **`02_improved_eval_report.ipynb`** to:
1. Load all 5 fold models
2. Compute OOF predictions + TTA
3. Per-fold AUC + mean +/- std
4. Temperature scaling + isotonic recalibration
5. Patient-level aggregation + comparison vs baseline

### If session timed out (some folds remaining):
1. This notebook's output is automatically saved
2. In the next session: add this output as input dataset
3. Update `PREV_CHECKPOINT_DIR` to point to it
4. Re-run all cells — completed folds are skipped automatically

### Outputs saved to `/kaggle/working/`:
- `fold_splits.json` — deterministic fold assignments
- `best_model_fold{0-4}.pth` — best model weights per fold
- `metrics_fold{0-4}.csv` — epoch-by-epoch metrics per fold
- `fold_training_curves.png` — visualization